### Download, Edit, and Upload Stream Manager datasets 

In [1]:
import json,os,requests
from cruciblelib import curated_read_controller, curated_write_controller, \
    data_catalog_controller, read_controller, write_controller, \
    utils, write_controller,authenticator
import pandas as pd
from datetime import timezone as tz
from datetime import datetime as dt
import shutil,logging

print()
print(' Environment Variables:')

print(' CRUCIBLE_SERVICES_HOST = ',os.getenv('CRUCIBLE_SERVICES_HOST'))
print(' CRUCIBLE_CERT_PATH = ',os.getenv('CRUCIBLE_CERT_PATH'))
print()
rc = read_controller.ReadController()
wc = write_controller.WriteController()
dcc = data_catalog_controller.DataCatalogController()
auth = authenticator.Authenticator(client_id=os.environ['CRUCIBLE_CLIENT_ID'],
                                      client_secret=os.environ['CRUCIBLE_CLIENT_SECRET'],
                                      grant_type=os.environ['CRUCIBLE_GRANT_TYPE'],
                                      token_url=os.environ['CRUCIBLE_TOKEN_URL'],
                                      log_level=logging.INFO,
                                      tls_cert=os.environ['CRUCIBLE_CERT_PATH'])


config_base_dir='stream_manager_config_files'
script_base_dir='stream_manager_functions'

if not os.path.exists(config_base_dir):
    os.makedirs(config_base_dir)
if not os.path.exists(script_base_dir):
    os.makedirs(script_base_dir)

def get_current_timestamp_string():
    """
    Gets the current time in UTC and formats it in the way dictated by the schema.

    Returns:
         str
    """
    # Get the current time in UTC and format it in the way dictated by the schema
    datetime_format_string = "%Y-%m-%dT%H:%M:%S.%f"
    current_time = dt.now(tz=tz.utc).strftime(datetime_format_string)[:-3] + 'Z'
    return current_time

def write_local_functions_to_crucible(dataset_name='Stream_Manager_Functions'):
    
    dcc.token = auth.get_token()
    wc.token = auth.get_token()
    # dcc.delete_dataset_by_name(dataset_name)
    # dcc.create_dataset(dataset_name,'script_name',type='ENTITY')
    # dcc.clear_dataset_by_name(dataset_name)

    timestamp_str=get_current_timestamp_string()

    #iterate through files in script_base_dir
    for file_name in os.listdir(script_base_dir):
        if 'py' in file_name and not 'ipynb' in file_name:
            full_file_name = script_base_dir + '/' + file_name

            print('trying to upload ',full_file_name)
  
            config_JSON = {}
            script_name = file_name.replace('.py','')

            config_JSON['script_name'] = script_name
  
            #open corresponding python file
            with open(script_base_dir+'/'+config_JSON['script_name']+'.py','r') as f:
                config_JSON['script_body'] = f.read()

            wc.upsert_by_name(dataset_name,[config_JSON])

def write_local_config_files_to_crucible(dataset_name='Stream_Manager_Configuration'):

    dcc.token = auth.get_token()
    wc.token = auth.get_token()
    # dcc.delete_dataset_by_name(dataset_name)
    # dcc.create_dataset(dataset_name,'origin_dataset',type='ENTITY')
    # dcc.clear_dataset_by_name(dataset_name)

    timestamp_str=get_current_timestamp_string()

    #get list of directories inside of config_base_dir, exclude files
    config_directories=os.listdir(config_base_dir)

    for dir in config_directories:
        if os.path.isdir(config_base_dir+'/'+ dir):
            #get list of files in each directory
            for file_name in os.listdir( config_base_dir+'/'+ dir):
                if 'json' in file_name:
                    full_file_name = config_base_dir+'/'+ dir + '/' + file_name
                    print('trying to upload ',full_file_name)
                    with open(full_file_name,'r') as f:
                        config_JSON = json.load(f)
                    if "origin_dataset" in config_JSON.keys():
                        # first remove legacy fields
                        config_JSON.pop("updatedTimestamp",None)
                        config_JSON.pop("creationTimestamp",None)
                        wc.upsert_by_name(dataset_name,[config_JSON])

def clear_local_Stream_Manager_files():
    # recursively delete all files in the config directory

    # in config directory, remove subdirectories and files within them
    for root, dirs, files in os.walk(config_base_dir):
        for dir in dirs:
            shutil.rmtree(os.path.join(root, dir))

    # remove files in script subdirectory
    for root, dirs, files in os.walk(script_base_dir):
        for file in files:
            os.remove(os.path.join(root, file))

    # remove subdirectories (if needed)
    for root, dirs, files in os.walk(script_base_dir):
        for dir in dirs:
            shutil.rmtree(os.path.join(root, dir))

def download_Stream_Manager_files():

    rc.token = auth.get_token()
    dataset_name='Stream_Manager_Configuration'
    dict_list=rc.search('select * from '+dataset_name)
    #write each config to a file
    for config_dict in dict_list:
        config_dict.pop('crucibleHeader',None)
        config_dict.pop("updatedTimestamp",None)
        config_dict.pop("creationTimestamp",None)
        POV_dir=config_dict['perspective']
        # see if directory exists
        if not os.path.exists(config_base_dir+'/'+POV_dir):
            os.makedirs(config_base_dir+'/'+POV_dir)
        with open(config_base_dir+'/'+POV_dir+'/'+config_dict['origin_dataset']+'.json', 'w') as outfile:
            json.dump(config_dict, outfile, indent=4)


    dataset_name='Stream_Manager_Functions'
    dict_list=rc.search('select * from '+dataset_name)
    #write each config and script to a file 
    for config_dict in dict_list:
        config_dict.pop('crucibleHeader',None)
        config_dict.pop("updatedTimestamp",None)
        config_dict.pop("creationTimestamp",None)
        with open(script_base_dir+'/'+config_dict['script_name']+'.py', 'w') as outfile:
            outfile.write(config_dict['script_body'])



 Environment Variables:
 CRUCIBLE_SERVICES_HOST =  crucibledev3.koverse.com
 CRUCIBLE_CERT_PATH =  /Users/cassidyta/certs/dev3ca.crt



### Download Stream Manager configuration files and functions
## STOP:
### only do this is you're certain you don't need the local config files

In [2]:
clear_local_Stream_Manager_files()
download_Stream_Manager_files()

### ________________________________________________________________________________________________________
## STOP
### You can now edit the downloaded scripts and JSON config files in Jupyter Lab before upserting the edits
### ________________________________________________________________________________________________________

### Upload local JSON config files to dataset Stream_Manager_Configurations
#### This will overwrite configs on Crucible via upsert

In [3]:
write_local_config_files_to_crucible('Stream_Manager_Configuration')

trying to upload  stream_manager_config_files/test_POV/test_feed_1.json
trying to upload  stream_manager_config_files/test_POV/test_feed_6.json
trying to upload  stream_manager_config_files/test_POV/test_feed_7.json
trying to upload  stream_manager_config_files/test_POV/test_feed_4.json
trying to upload  stream_manager_config_files/test_POV/test_feed_8.json
trying to upload  stream_manager_config_files/test_POV/test_feed_9.json
trying to upload  stream_manager_config_files/test_POV/test_feed_5.json
trying to upload  stream_manager_config_files/test_POV/test_feed_2.json
trying to upload  stream_manager_config_files/test_POV/test_feed_3.json
trying to upload  stream_manager_config_files/Live_POV/Live_POV_perspective_config.json
trying to upload  stream_manager_config_files/Live_POV/AIS_U.json
trying to upload  stream_manager_config_files/Live_POV/Open_Sky.json
trying to upload  stream_manager_config_files/Live_POV/ADSB_U.json
trying to upload  stream_manager_config_files/Live_POV/Seattle

### Write local Functions to dataset Stream_Manager_Functions
#### This will overwrite scripts on Crucible via upsert

In [4]:
write_local_functions_to_crucible('Stream_Manager_Functions')

trying to upload  stream_manager_functions/custom_functions_perf.py
trying to upload  stream_manager_functions/unit_conversions.py
trying to upload  stream_manager_functions/test_pydantic.py
trying to upload  stream_manager_functions/test_object_enrichment.py
trying to upload  stream_manager_functions/custom_functions.py
trying to upload  stream_manager_functions/object_perf_enrichment.py
trying to upload  stream_manager_functions/test_custom_functions.py
trying to upload  stream_manager_functions/rosetta_lookup_functions.py
trying to upload  stream_manager_functions/test_unit_conversions.py
trying to upload  stream_manager_functions/object_enrichment.py


### Test downloaded scripts

In [ ]:
#download and import unit_conversions.py
dict_list=rc.search("select * from Stream_Manager_Functions")

for d in dict_list:
    if d['script_name']=='unit_conversions':
        with open('unit_conversions.py','w') as f:
            f.write(d['script_body'])
# delete unit_conversions.py from the file system
import unit_conversions
os.remove('unit_conversions.py')
unit_conversions.ft_to_m(1)
print('this should be about 0.3048:   ')
print('         ',unit_conversions.ft_to_m(1))



### Scratch code


#### Example here kept here kept for reference

In [ ]:
dcc.token = auth.get_token()
dcc.clear_dataset_by_name('Live_POV_Objects')
dcc.clear_dataset_by_name('Live_POV_PrincipalTrackHeads')
dcc.clear_dataset_by_name('Live_POV_PrincipalTrackEvents')
dcc.clear_dataset_by_name('Live_POV_ComponentTrackHeads')
dcc.clear_dataset_by_name('Live_POV_ComponentTrackEvents')
dcc.clear_dataset_by_name('Live_POV_ObjectManagementEvents')
dcc.clear_dataset_by_name('Live_POV_SupersededObjects')
dcc.clear_dataset_by_name('Live_POV_Object_Events')

In [3]:
perspective_names=['Live',]
live_indices = [i for i, name in enumerate(perspective_names) if 'live' in name.lower()]
print(live_indices )

[0]
